# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. [1,  2,  3] [0 1 1]
2. [4, 4, 4]
3. [[0 1 2]
 [1 2 3]
 [2 3 4]]
4. 6.0
5. 7.5

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [1]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [2]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [ ]:
# 1

amount_c = amount[clean]
month_c  = month[clean]
cat_c    = cat[clean]

clean_count = clean.sum()

rev_sep = amount_c[month_c == "2026-09"].sum()
rev_oct = amount_c[month_c == "2026-10"].sum()
pct_change = (rev_oct - rev_sep) / rev_sep * 100

print(f"Строк в clean: {clean_count}")
print(f"Выручка за сентябрь: {rev_sep}")
print(f"Выручка за октябрь: {rev_oct}")
print(f"Изменение: {pct_change}%\n")

Строк в clean: 4739
Выручка за сентябрь: 35243410.0
Выручка за октябрь: 34419520.0
Изменение: -2.337713632137185%



## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [6]:
# 2
categories = np.unique(cat)      
months = np.unique(month)        

rev_table = np.array([
    [amount_c[(cat_c == c) & (month_c == m)].sum() for m in months]
    for c in categories
])

for c, row in zip(categories, rev_table):
    print(f"{c}: {row[0]}, {row[1]}")

total_by_cat = rev_table.sum(axis=1)
top_cat = categories[total_by_cat.argmax()]

cat_pct_change = (rev_table[:, 1] - rev_table[:, 0]) / rev_table[:, 0] * 100
worst_cat_idx = cat_pct_change.argmin()
worst_cat = categories[worst_cat_idx]

print(f"\nБольше всего выручки за 2 месяца: {top_cat} ({total_by_cat.max()})")
print(f"Сильнее всех упала: {worst_cat} ({cat_pct_change[worst_cat_idx]}%)\n")

Дом: 4997730.0, 5125790.0
Книги: 1065300.0, 1140910.0
Красота: 2610950.0, 2678990.0
Одежда: 9167100.0, 8505730.0
Электроника: 17402330.0, 16968100.0

Больше всего выручки за 2 месяца: Электроника (34370430.0)
Сильнее всех упала: Одежда (-7.214604400519248%)



## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [8]:
# 3
orders_table = np.array([
    [((cat_c == c) & (month_c == m)).sum() for m in months]
    for c in categories
])

avg_check_table = rev_table / orders_table

oct_avg_check = avg_check_table[:, 1]
max_check_idx = oct_avg_check.argmax()

for c, check in zip(categories, oct_avg_check):
    print(f"  {c}: {check}")

print(f"\nСамый высокий средний чек в октябре: {categories[max_check_idx]} {oct_avg_check[max_check_idx]}")

  Дом: 10768.466386554623
  Книги: 3815.752508361204
  Красота: 6158.597701149425
  Одежда: 13045.598159509202
  Электроника: 33600.19801980198

Самый высокий средний чек в октябре: Электроника 33600.19801980198


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [ ]:
# 4

## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [ ]:
# 5

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [ ]:
# 6

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [ ]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8